# ResNet-18 Training on CIFAR-10 / CIFAR-100
### Master's Research: Machine Unlearning — Multi-Class Image Classification

This notebook is an experiment environment with baseline implementation for my master's thesis topic related to the machine unlearning paradigm. The analysis of advanced machine unlearning methods for multi-class image classification models.
Mikolaj Hajder 264478

Datasets that are going to be used in the analysis:
1. CIFAR-10 -> CIFAR-10 is a well-known benchmark dataset in the field of machine learning, containing 60,000 32x32 color images across 10 classes. Each class contains 6,000 images.
The dataset is split into a training set of 50,000 images and a test set of 10,000 images, with an equal number of images per class in both sets.


2. CIFAR-100 -> CIFAR-100 has 100 classes containing 600 images each. There are 500 training images and 100 testing images per class.


## 1. Imports & Reproducibility

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time

# ── Shared utilities ──────────────────────────────────────────────────────────
# build_resnet18      : constructs the CIFAR-adapted ResNet-18
# evaluate            : runs inference on a DataLoader, returns (loss, accuracy)
# per_class_accuracy  : returns per-class accuracy tensor
# save_checkpoint     : saves model weights + metadata to a .pth file
# load_checkpoint     : loads a .pth file and returns a ready model
# STATS               : normalisation mean/std for CIFAR-10 and CIFAR-100
from utils import (build_resnet18, evaluate, per_class_accuracy,
                   save_checkpoint, load_checkpoint, STATS)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {DEVICE}")

## 2. Configuration

Change `DATASET` to `"CIFAR10"` or `"CIFAR100"` to switch datasets.
Everything below this cell adapts automatically.

In [ ]:
# ── Experiment config ─────────────────────────────────────────────────────────
DATASET        = "CIFAR10"    # "CIFAR10" | "CIFAR100"
DATA_ROOT      = "./data"
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Training hyper-parameters
BATCH_SIZE    = 128
NUM_EPOCHS    = 100
LR            = 0.1
MOMENTUM      = 0.9
WEIGHT_DECAY  = 5e-4

# LR schedule — multiply LR by LR_GAMMA at each milestone epoch
LR_MILESTONES = [50, 75]
LR_GAMMA      = 0.1

NUM_CLASSES = 10 if DATASET == "CIFAR10" else 100
print(f"Dataset  : {DATASET}  |  Classes: {NUM_CLASSES}")

## 3. Data Loading & Augmentation

Normalisation statistics (mean and std per colour channel) are imported
from `utils.STATS` — the same values will be used in every unlearning notebook
to guarantee consistent preprocessing.

In [ ]:
# ── Normalisation stats from utils ────────────────────────────────────────────
mean, std = STATS[DATASET]["mean"], STATS[DATASET]["std"]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# ── Load datasets ─────────────────────────────────────────────────────────────
DatasetClass = (torchvision.datasets.CIFAR10 if DATASET == "CIFAR10"
                else torchvision.datasets.CIFAR100)

train_dataset = DatasetClass(root=DATA_ROOT, train=True,  download=True,
                              transform=train_transform)
test_dataset  = DatasetClass(root=DATA_ROOT, train=False, download=True,
                              transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples : {len(train_dataset):,}")
print(f"Test  samples : {len(test_dataset):,}")
print(f"Batches/epoch : {len(train_loader)}")

### 3.1 Quick Sanity-Check — Sample Images

In [ ]:
def imshow(img_tensor, title=None):
    """Denormalise and display a single image tensor."""
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t  = torch.tensor(std).view(3, 1, 1)
    img = (img_tensor * std_t + mean_t).permute(1, 2, 0).numpy().clip(0, 1)
    plt.imshow(img)
    if title:
        plt.title(title, fontsize=8)
    plt.axis("off")


images, labels = next(iter(train_loader))
classes = train_dataset.classes

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    plt.sca(ax)
    imshow(images[i], title=classes[labels[i]])
plt.suptitle(f"{DATASET} — sample training images", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 4. Model — ResNet-18

`build_resnet18` is imported from `utils.py`.
It replaces the standard ImageNet 7x7/stride-2 stem with a 3x3/stride-1 conv
and removes the max-pool, which is the standard modification for 32x32 CIFAR images
used throughout the machine-unlearning literature.

In [ ]:
# build_resnet18 is defined in utils.py — imported in cell 1
model = build_resnet18(NUM_CLASSES).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Optimiser & LR Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=True,
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

print("Criterion :", criterion)
print("Optimizer :", optimizer)
print("Scheduler :", scheduler)

## 6. Training Helper

`evaluate` is imported from `utils.py` and shared with all unlearning notebooks.
`train_one_epoch` is defined locally because it is only needed during training —
unlearning notebooks do not use it.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Run one full pass over the training loader and update model weights.
    Returns (avg_loss, accuracy_percent) for this epoch.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()                  # clear gradients from previous batch
        outputs = model(images)                # forward pass
        loss    = criterion(outputs, labels)   # compute loss
        loss.backward()                        # backpropagation
        optimizer.step()                       # update weights

        total_loss += loss.item() * images.size(0)
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += images.size(0)

    return total_loss / total, 100.0 * correct / total


# evaluate() is imported from utils.py — no need to redefine it here
print("train_one_epoch defined.  evaluate imported from utils.py.")

## 7. Training Loop

In [ ]:
history = {"train_loss": [], "train_acc": [],
           "test_loss":  [], "test_acc":  []}

best_acc   = 0.0
best_ckpt  = os.path.join(CHECKPOINT_DIR,
                          f"resnet18_{DATASET.lower()}_best.pth")
final_ckpt = os.path.join(CHECKPOINT_DIR,
                          f"resnet18_{DATASET.lower()}_final.pth")

print(f"{'Epoch':>6}  {'LR':>8}  "
      f"{'Train Loss':>10}  {'Train Acc':>9}  "
      f"{'Test Loss':>9}  {'Test Acc':>8}  {'Time':>6}")
print("-" * 72)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                      criterion, optimizer, DEVICE)
    te_loss, te_acc = evaluate(model, test_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss)
    history["test_acc"].append(te_acc)

    current_lr = scheduler.get_last_lr()[0]
    elapsed    = time.time() - t0

    print(f"{epoch:>6}  {current_lr:>8.5f}  "
          f"{tr_loss:>10.4f}  {tr_acc:>8.2f}%  "
          f"{te_loss:>9.4f}  {te_acc:>7.2f}%  {elapsed:>5.1f}s")

    # ── Save best checkpoint via utils.save_checkpoint ────────────────────────
    if te_acc > best_acc:
        best_acc = te_acc
        save_checkpoint(
            model, best_ckpt,
            epoch=epoch, test_acc=te_acc,
            dataset=DATASET, num_classes=NUM_CLASSES,
            extra={"optim_state": optimizer.state_dict()}
        )

# ── Save final checkpoint ─────────────────────────────────────────────────────
save_checkpoint(
    model, final_ckpt,
    epoch=NUM_EPOCHS, test_acc=te_acc,
    dataset=DATASET, num_classes=NUM_CLASSES,
    extra={"optim_state": optimizer.state_dict()}
)

print(f"\nTraining complete.  Best test accuracy : {best_acc:.2f}%")
print(f"Best checkpoint  : {best_ckpt}")
print(f"Final checkpoint : {final_ckpt}")

## 8. Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history["train_loss"], label="Train", linewidth=2)
axes[0].plot(epochs, history["test_loss"],  label="Test",  linewidth=2)
for m in LR_MILESTONES:
    axes[0].axvline(x=m, color="grey", linestyle="--", alpha=0.6)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title(f"{DATASET} — Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["train_acc"], label="Train", linewidth=2)
axes[1].plot(epochs, history["test_acc"],  label="Test",  linewidth=2)
for m in LR_MILESTONES:
    axes[1].axvline(x=m, color="grey", linestyle="--", alpha=0.6)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title(f"{DATASET} — Accuracy (best: {best_acc:.2f}%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR,
            f"resnet18_{DATASET.lower()}_training_curves.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 9. Per-Class Accuracy

`per_class_accuracy` is imported from `utils.py`.

In [ ]:
# per_class_accuracy is imported from utils.py — no local definition needed
per_class_acc = per_class_accuracy(model, test_loader, NUM_CLASSES, DEVICE)

fig_h = max(6, NUM_CLASSES * 0.22)
fig, ax = plt.subplots(figsize=(10, fig_h))

y_pos = np.arange(NUM_CLASSES)
ax.barh(y_pos, per_class_acc.numpy(), align="center", color="steelblue")
ax.set_yticks(y_pos)
ax.set_yticklabels(train_dataset.classes, fontsize=8)
ax.set_xlabel("Accuracy (%)")
ax.set_title(f"{DATASET} — Per-Class Test Accuracy")
ax.axvline(x=per_class_acc.mean().item(), color="red", linestyle="--",
           label=f"Mean = {per_class_acc.mean():.1f}%")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR,
            f"resnet18_{DATASET.lower()}_per_class_acc.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'Class':<20} {'Accuracy':>8}")
print("-" * 30)
for name, acc in zip(train_dataset.classes, per_class_acc.tolist()):
    print(f"{name:<20} {acc:>7.2f}%")
print("-" * 30)
print(f"{'Mean':<20} {per_class_acc.mean():>7.2f}%")

## 10. Load Checkpoint (Optional)

`load_checkpoint` is imported from `utils.py`.
Use this cell in a fresh session to reload a saved model without retraining.

In [ ]:
# load_checkpoint is imported from utils.py — no local definition needed

# Example usage (uncomment to run):
# loaded_model = load_checkpoint(best_ckpt, DEVICE)

## 11. Summary

| Item | Value |
|------|-------|
| Dataset | `DATASET` |
| Architecture | ResNet-18 (CIFAR-adapted stem) |
| Epochs | 100 |
| Optimiser | SGD + Nesterov momentum |
| LR schedule | MultiStep (x0.1 @ epoch 50, 75) |
| Best test acc | see cell 7 output |
| Checkpoints | `./checkpoints/` |
| Shared code | `utils.py` |

The saved checkpoints (`*_best.pth`, `*_final.pth`) contain the full model state
and are ready to be loaded by all unlearning notebooks via `load_checkpoint` from `utils.py`.